# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook verifies data quality using PyDeequ and routes invalid records to a Dead Letter Queue (DLQ).

In [ ]:
import os
os.environ["SPARK_VERSION"] = "3.4"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day06_DQ_DLQ") \
    .config("spark.jars.packages", "com.amazon.deequ:deequ:2.0.7-spark-3.4") \
    .getOrCreate()


In [ ]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite

# Create synthetic valid and invalid data
data = [
    (1, "pat_1", 12345, "2023-01-01", "J45"), # Valid
    (2, None, 54321, "2023-01-02", "E11"),    # Invalid: missing person_id
    (3, "pat_3", -1, "2023-01-03", "I10")     # Invalid: negative concept_id
]
df = spark.createDataFrame(data, ["condition_occurrence_id", "person_id", "condition_concept_id", "condition_start_date", "condition_source_value"])

print("Input Data:")
df.show()


In [ ]:
check = Check(spark, CheckLevel.Error, "DQ Checks")

checkResult = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(
        check.isComplete("person_id") \
        .isNonNegative("condition_concept_id")
    ) \
    .run()

checkResult_df = VerificationSuite.checkResultsAsDataFrame(spark, checkResult)
print("DQ Results:")
checkResult_df.show(truncate=False)


In [ ]:
# Row-level routing simulation
from pyspark.sql.functions import col, lit

dlq_df = df.filter(col("person_id").isNull() | (col("condition_concept_id") < 0))
valid_df = df.filter(col("person_id").isNotNull() & (col("condition_concept_id") >= 0))

print("Routed to Valid Table:")
valid_df.show()

print("Routed to DLQ Table (with manifest logic to be added):")
dlq_df = dlq_df.withColumn("error_reason", lit("Failed DQ Check (Missing person_id or negative concept_id)"))
dlq_df.show(truncate=False)
